# Exercise 5.1: Global Evaluation

Generate `dev.txt` and `test.txt` prediction files for the official
SemEval dev set and test set using the best model from Experiment O
(ModernBERT-large verbalizer).

**Output format:** One prediction per line (`0` = No PCL, `1` = PCL).

In [ ]:
import os
import sys
import json
import logging

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import classification_report

sys.path.insert(0, ".")
from utils.data import load_data, clean_text
from utils.pcl_dataset import PCLVerbalizerDataset
from utils.pcl_deberta import PCLVerbalizer
from utils.eval import evaluate

SEED = 42
DATA_DIR = "data"
OUT_DIR = "out"
MODEL_OUT_DIR = OUT_DIR
EXP_NAME = "O_verbalizer"
MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 256
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")

if os.getlogin() == "jc4423":
    LOG.info("Running on lab machines, changing caches and model out dir")
    os.environ["HF_HOME"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["TRANSFORMERS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["HF_DATASETS_CACHE"] = "/vol/bitbucket/jc4423/.cache/huggingface"
    os.environ["PIP_CACHE_DIR"] = "/vol/bitbucket/jc4423/.cache/pip"
    os.environ["XDG_CACHE_HOME"] = "/vol/bitbucket/jc4423/.cache"
    MODEL_OUT_DIR = "/vol/bitbucket/jc4423/models/"

## 1. Load Best Model Config

Read the saved hyperparameters from the best Optuna trial.

In [ ]:
with open(os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_params.json")) as f:
    best_config = json.load(f)

print(json.dumps(best_config, indent=2))

# Extract key params
BEST_THRESHOLD = best_config["best_threshold"]
TEMPLATE_IDX = best_config["template_idx"]
VERBALIZER_NAME = best_config["verbalizer"]

LOG.info(f"Best threshold: {BEST_THRESHOLD}")
LOG.info(f"Verbalizer: {VERBALIZER_NAME}, Template idx: {TEMPLATE_IDX}")

## 2. Setup Verbalizer & Tokeniser

In [ ]:
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verbalizer sets (must match training)
VERBALIZER_SETS = [
    ("Yes_No",     ["Yes"],  ["No"]),
    ("yes_no",     ["yes"],  ["no"]),
    ("True_False", ["True"], ["False"]),
    ("true_false", ["true"], ["false"]),
]

TEMPLATE_OPTIONS = [
    'Is the following text patronising or condescending? "{text}" Answer: {mask}',
    'Does the author of this text sound patronising or condescending? "{text}" Answer: {mask}',
    'Is this text talking down to people? "{text}" Answer: {mask}',
    'Does the following text contain patronising language? "{text}" Answer: {mask}',
]


def words_to_first_subword_ids(words: list[str]) -> list[int]:
    return [tokeniser.encode(w, add_special_tokens=False)[0] for w in words]


# Resolve verbalizer IDs and template
verb_set = [v for v in VERBALIZER_SETS if v[0] == VERBALIZER_NAME][0]
pos_ids = words_to_first_subword_ids(verb_set[1])
neg_ids = words_to_first_subword_ids(verb_set[2])
template = TEMPLATE_OPTIONS[TEMPLATE_IDX]

LOG.info(f"Verbalizer '{VERBALIZER_NAME}': pos_ids={pos_ids}, neg_ids={neg_ids}")
LOG.info(f"Template: {template}")

## 3. Load Best Model Weights

In [ ]:
model = PCLVerbalizer(
    pos_verbalizer_ids=pos_ids,
    neg_verbalizer_ids=neg_ids,
    model_name=MODEL_NAME,
    gradient_checkpointing=False,
    cache_dir="/vol/bitbucket/jc4423/.cache/huggingface" if os.getlogin() == "jc4423" else None,
).to(DEVICE)

state_dict = torch.load(
    os.path.join(MODEL_OUT_DIR, f"exp_{EXP_NAME}_best_model.pt"),
    map_location=DEVICE,
)
model.load_state_dict(state_dict)
model.eval()
LOG.info("Best model loaded.")

## 4. Generate Dev Predictions (`dev.txt`)

Use the official dev set (labels available) — also serves as a sanity check.

In [ ]:
# Load dev set manually (no dropna — keep all rows so prediction count matches)
col_names = ["par_id", "art_id", "keyword", "country_code", "text", "label"]
df = pd.read_csv(
    os.path.join(DATA_DIR, "dontpatronizeme_pcl.tsv"),
    sep="\t", skiprows=4, names=col_names, index_col="par_id",
)
df["binary_label"] = (df["label"] >= 2).astype(int)
df["text"] = df["text"].fillna("").apply(clean_text)

dev_ids = pd.read_csv(os.path.join(DATA_DIR, "dev_semeval_parids-labels.csv"))["par_id"].values
dev_df = df.loc[df.index.isin(dev_ids), ["text", "binary_label"]].copy()
LOG.info(f"Dev set: {len(dev_df)} samples, {dev_df['binary_label'].sum()} positive")

dev_ds = PCLVerbalizerDataset(
    texts=dev_df["text"].tolist(),
    labels=dev_df["binary_label"].tolist(),
    max_length=MAX_LENGTH,
    tokeniser=tokeniser,
    template=template,
)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

dev_metrics = evaluate(model, DEVICE, dev_loader, threshold=BEST_THRESHOLD)

print(f"Dev F1: {dev_metrics['f1']:.4f}")
print(f"Dev Precision: {dev_metrics['precision']:.4f}")
print(f"Dev Recall: {dev_metrics['recall']:.4f}")
print()
print(classification_report(
    dev_metrics["labels"], dev_metrics["preds"],
    target_names=["Non-PCL", "PCL"],
))

# Write dev.txt
dev_preds = dev_metrics["preds"]
with open("dev.txt", "w") as f:
    for p in dev_preds:
        f.write(f"{p}\n")

LOG.info(f"dev.txt written: {len(dev_preds)} predictions")
assert len(dev_preds) == len(dev_df), f"Expected {len(dev_df)} but got {len(dev_preds)}"

## 5. Generate Test Predictions (`test.txt`)

Load the unlabelled test set (`task4_test.tsv`) and generate predictions.

In [ ]:
# Load test set (no labels)
test_df = pd.read_csv(
    os.path.join(DATA_DIR, "task4_test.tsv"),
    sep="\t",
    header=None,
    names=["par_id", "art_id", "keyword", "country_code", "text"],
    index_col="par_id",
)
test_df["text"] = test_df["text"].apply(clean_text)
LOG.info(f"Test set: {len(test_df)} samples")
test_df.head()

In [ ]:
# Create test dataset (no labels)
test_ds = PCLVerbalizerDataset(
    texts=test_df["text"].tolist(),
    labels=None,
    max_length=MAX_LENGTH,
    tokeniser=tokeniser,
    template=template,
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Run inference
model.eval()
all_scores = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        mask_token_indices = batch["mask_token_indices"].to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_token_indices=mask_token_indices,
        ).squeeze(-1)
        all_scores.append(logits.cpu())

all_scores = torch.cat(all_scores)
probs = torch.sigmoid(all_scores)
test_preds = (probs >= BEST_THRESHOLD).long().numpy()

LOG.info(f"Test predictions: {len(test_preds)} total, "
         f"{test_preds.sum()} positive ({test_preds.mean():.2%})")

# Write test.txt
with open("test.txt", "w") as f:
    for p in test_preds:
        f.write(f"{p}\n")

LOG.info(f"test.txt written: {len(test_preds)} predictions")
assert len(test_preds) == 3832, f"Expected 3832 but got {len(test_preds)}"

## 6. Sanity Checks

In [ ]:
# Verify files
for fname, expected_len in [("dev.txt", len(dev_df)), ("test.txt", 3832)]:
    with open(fname) as f:
        lines = f.readlines()
    n_lines = len(lines)
    values = set(line.strip() for line in lines)
    print(f"{fname}: {n_lines} lines, values: {values}")
    assert n_lines == expected_len, f"{fname}: expected {expected_len} lines, got {n_lines}"
    assert values <= {"0", "1"}, f"{fname}: unexpected values {values - {'0', '1'}}"

print("\nAll checks passed!")